# CS781 Phase 1 - four architectures on the starter-code framework

## 1. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("hf_token")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
  print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import shutil


# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# You can use any of the 4 train files or all the files for pre-training the legal model
TRAIN_FILES = [
  "Phase_1_Pretraining/Train/train_acts.json",
  "Phase_1_Pretraining/Train/train_high_courts.json",
  "Phase_1_Pretraining/Train/train_supreme_court.json",
  "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# ---- which architecture this run trains: "gpt", "modern", "rnn" or "t5" ----
MODEL_TYPE = "modern"   # best model; run once per type: "gpt", "modern", "rnn", "t5"

SMOKE_TEST = False

# Training configuration
TRAIN_TOKENS = 120_000_000      # total over all TRAIN_FILES (split equally: 30M per file)
VAL_TOKENS = 5_000_000
EPOCHS = 1
SEED = 42
random.seed(SEED)
MODEL_CONTEXT_LENGTH = 2048

# --- gpt / modern (decoder-only Transformers) ---
N_EMBD = 640
N_LAYER = 10
N_HEAD = 10
N_KV_HEAD = 5            # modern only: grouped-query attention, 2 query heads share 1 key/value head
D_FF = 4 * N_EMBD        # feed-forward hidden size, 4 x d_model (as in BERT / GPT)
DROPOUT = 0.1

# --- rnn ---
RNN_CELL = "lstm"        # "lstm" or "gru"
RNN_HIDDEN = 1152        # use 1280 for "gru" to stay above 50M parameters
RNN_LAYERS = 4
RNN_BLOCK_SIZE = 256     # truncated BPTT length (BPTT-k, Module 2.1)

# --- t5 ---
T5_D_MODEL = 512
T5_ENC_LAYERS = 8
T5_DEC_LAYERS = 8
T5_HEADS = 8
T5_D_FF = 2048
T5_ENC_LEN = 1024        # encoder reads (the end of) the prompt
T5_DEC_LEN = 384         # decoder writes the continuation (= MAX_GENERATION_TOKENS)
T5_N_SENTINELS = 100     # <extra_id_0> ... <extra_id_99> for span corruption
T5_SPAN_FRACTION = 0.5   # first half of the training blocks: span-corruption pre-training,
                         # second half: prompt -> continuation fine-tuning (Module 2.4 T5 pipeline)
T5_FINETUNE_LR = 1e-3    # Module 2.4: T5 fine-tuning uses a constant learning rate (often 0.001)

# --- Trainer settings for each architecture ---
MAX_GRAD_NORM = 1.0      # Trainer default (gradient clipping)
if MODEL_TYPE == "gpt" or MODEL_TYPE == "modern":
  TRAIN_BLOCK_SIZE = MODEL_CONTEXT_LENGTH
  BATCH_SIZE = 2         # starter used 4 x 2 accumulation; our hand-written attention keeps the
  GRAD_ACCUM = 4         # full T x T score matrix in memory, so 2 x 4 (same 8 sequences/step) fits a T4
  LEARNING_RATE = 3e-4
elif MODEL_TYPE == "rnn":
  TRAIN_BLOCK_SIZE = RNN_BLOCK_SIZE
  BATCH_SIZE = 32
  GRAD_ACCUM = 1
  LEARNING_RATE = 1e-3   # Assignment 2 used lr 1e-3 for the LSTM / GRU models
  MAX_GRAD_NORM = 5.0    # Assignment 2 used clip_grad_norm_(max_norm=5.0) for RNNs
elif MODEL_TYPE == "t5":
  TRAIN_BLOCK_SIZE = T5_ENC_LEN + T5_DEC_LEN
  BATCH_SIZE = 4
  GRAD_ACCUM = 2
  LEARNING_RATE = 3e-4
else:
  raise ValueError("MODEL_TYPE must be gpt, modern, rnn or t5")

if SMOKE_TEST:
  TRAIN_TOKENS = 2_000_000
  VAL_TOKENS = 200_000
  N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD, D_FF = 128, 2, 4, 2, 512
  RNN_HIDDEN, RNN_LAYERS = 128, 2
  T5_D_MODEL, T5_ENC_LAYERS, T5_DEC_LAYERS, T5_HEADS, T5_D_FF = 128, 2, 2, 4, 512

HELDOUT_N = 5 if SMOKE_TEST else 50      # held-out documents PER FILE for local evaluation

def heldout_indices(records, n=None, seed=None):
  n = HELDOUT_N if n is None else n
  rng = random.Random(SEED if seed is None else seed)
  eligible = [i for i, r in enumerate(records) if len(r["text"].split()) >= 640]
  return set(rng.sample(eligible, min(n, len(eligible))))

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
DECODING_STRATEGY = "top_p"   # "greedy", "sample", "top_k" or "top_p"  (Module 2.1, Assignment 2)
TEMPERATURE = 0.9      # best setting from the Codabench sweep
TOP_K = 10
TOP_P = 0.9
GENERATION_BATCH_SIZE = 8     # starter used 32; smaller because our attention is not memory-optimised

# Output (Google Drive). Every architecture gets its own folder / files.
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, f"legal-{MODEL_TYPE}-final")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, f"prediction_{MODEL_TYPE}.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, f"submission_{MODEL_TYPE}.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, f"training_validation_loss_{MODEL_TYPE}.png")
os.makedirs(OUTPUT_DIR, exist_ok=True)
LOCAL_TRUTH_PATH = os.path.join(OUTPUT_DIR, "local_truth.json")
LOCAL_PROMPTS_PATH = os.path.join(OUTPUT_DIR, "local_prompts.json")
LOCAL_PREDICTION_PATH = os.path.join(OUTPUT_DIR, f"local_prediction_{MODEL_TYPE}.json")
LOCAL_BASELINE_PATH = os.path.join(OUTPUT_DIR, "local_baseline_prediction.json")
print("Configuration loaded. MODEL_TYPE =", MODEL_TYPE)

## 4. Load Tokenizer

In [ ]:
import json
import heapq
from collections import Counter, defaultdict

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
WORD_END = "</w>"

class ByteLevelBPE:
  def __init__(self, vocab_size):
      self.vocab_size = vocab_size
      self.merges = []
      self.merge_ranks = {}
      self.itos = []
      self.stoi = {}
      self._cache = {}
      self.pad_id = None
      self.bos_id = None
      self.eos_id = None

  def _word_to_symbols(self, word):
      byte_vals = word.encode("utf-8")
      symbols = [chr(b) for b in byte_vals]
      symbols.append(WORD_END)
      return symbols

  def train(self, sentences):
      word_freq = Counter()
      for sent in sentences:
          for word in sent.split():
              word_freq[word] += 1

      words = list(word_freq)
      freqs = [word_freq[w] for w in words]
      word_symbols = [self._word_to_symbols(w) for w in words]

      vocab_list = list(SPECIALS)
      vocab_seen = set(vocab_list)

      initial_symbols = {chr(b) for b in range(256)} | {WORD_END}

      for s in sorted(initial_symbols):
          if s not in vocab_seen:
              vocab_list.append(s)
              vocab_seen.add(s)

      # this loop recounts every pair every merge, which is far too slow
      # for the legal corpus. Replace it with incremental pair counts + pair_to_words.
      pair_counts = Counter()
      pair_to_words = defaultdict(set)
      for wi, symbols in enumerate(word_symbols):
          f = freqs[wi]
          for i in range(len(symbols) - 1):
              p = (symbols[i], symbols[i + 1])
              pair_counts[p] += f
              pair_to_words[p].add(wi)

      heap = [(-c, p) for p, c in pair_counts.items()]
      heapq.heapify(heap)

      while len(vocab_list) < self.vocab_size:
          best_pair, best_count = None, 0
          while heap:
              negc, p = heapq.heappop(heap)
              if pair_counts.get(p, 0) == -negc and -negc > 0:
                  best_pair, best_count = p, -negc
                  break
          if best_pair is None or best_count <= 1:
              break

          a, b = best_pair
          merged = a + b
          changed = set()

          for wi in list(pair_to_words[best_pair]):
              symbols = word_symbols[wi]
              f = freqs[wi]

              old_pairs = Counter(zip(symbols, symbols[1:]))   # positions, so overlaps count
              new_symbols = []
              i = 0
              while i < len(symbols):
                  if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                      new_symbols.append(merged)
                      i += 2
                  else:
                      new_symbols.append(symbols[i])
                      i += 1
              word_symbols[wi] = new_symbols
              new_pairs = Counter(zip(new_symbols, new_symbols[1:]))

              for p, c in old_pairs.items():
                  pair_counts[p] -= c * f
                  changed.add(p)
                  if p not in new_pairs:
                      pair_to_words[p].discard(wi)
              for p, c in new_pairs.items():
                  pair_counts[p] += c * f
                  changed.add(p)
                  pair_to_words[p].add(wi)

          for p in changed:
              c = pair_counts[p]
              if c > 0:
                  heapq.heappush(heap, (-c, p))
              else:
                  del pair_counts[p]
                  pair_to_words.pop(p, None)

          self.merges.append([a, b])
          self.merge_ranks[(a, b)] = len(self.merges) - 1

          if merged not in vocab_seen:
              vocab_list.append(merged)
              vocab_seen.add(merged)

      self.itos = vocab_list
      self.stoi = {tok: i for i, tok in enumerate(self.itos)}
      self.pad_id = self.stoi["<pad>"]
      self.bos_id = self.stoi["<s>"]
      self.eos_id = self.stoi["</s>"]
      return self

  def encode_word(self, word):
      if word in self._cache:
        return self._cache[word]
      symbols = self._word_to_symbols(word)
      while len(symbols) > 1:
          pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
          ranked = [(self.merge_ranks[p], i) for i, p in enumerate(pairs) if p in self.merge_ranks]
          if not ranked:
              break
          _, i = min(ranked)
          symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]

      self._cache[word] = symbols
      return symbols

  def encode_ids(self, sentence):
      unk = self.stoi["<unk>"]
      return [self.stoi.get(t, unk) for t in self.encode(sentence)]

  def decode(self, ids):
      text = "".join(self.itos[i] for i in ids if self.itos[i] not in SPECIALS)
      words = text.split(WORD_END)
      out = [bytes(ord(c) for c in w).decode("utf-8", errors="replace") for w in words if w]
      return " ".join(out)

  def encode(self, sentence):
      tokens = []
      for word in sentence.split():
          tokens.extend(self.encode_word(word))
      return tokens

  def save(self, path):
      with open(path, "w", encoding="utf-8") as f:
          json.dump({"merges": self.merges, "itos": self.itos,
                      "vocab_size": self.vocab_size}, f, ensure_ascii=False)

  @staticmethod
  def load_bpe(path):
      with open(path, "r", encoding="utf-8") as f:
          data = json.load(f)
      bpe = ByteLevelBPE(vocab_size=data["vocab_size"])
      bpe.merges = data["merges"]
      bpe.itos = data["itos"]
      bpe.stoi = {tok: i for i, tok in enumerate(bpe.itos)}
      bpe.merge_ranks = {(a, b): i for i, (a, b) in enumerate(bpe.merges)}
      bpe.pad_id = bpe.stoi["<pad>"]
      bpe.bos_id = bpe.stoi["<s>"]
      bpe.eos_id = bpe.stoi["</s>"]
      return bpe

In [ ]:
import gc
from huggingface_hub import hf_hub_download

# ---- YOUR CHOICES: justify these in the report ----
BPE_VOCAB_SIZE = 2000 if SMOKE_TEST else 16000   # target vocabulary size incl. the 4 special tokens
BPE_DOCS_PER_FILE = 50 if SMOKE_TEST else 150   # docs sampled per file
BPE_TRAIN_FILES = TRAIN_FILES
BPE_PATH = os.path.join(OUTPUT_DIR, "bpe_legal.json")

if os.path.exists(BPE_PATH) and not SMOKE_TEST:
  # already trained in an earlier run -> just load it (every architecture uses the SAME tokenizer)
  bpe = ByteLevelBPE.load_bpe(BPE_PATH)
  print("Loaded BPE from", BPE_PATH, "| vocab size:", len(bpe.itos))
else:
  random.seed(SEED)
  sample_texts = []
  for fname in BPE_TRAIN_FILES:
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type="dataset", token=HF_TOKEN)
    with open(p, "r", encoding="utf-8") as f:
      recs = json.load(f)
    held = heldout_indices(recs)   # keep held-out docs out of the BPE sample
    pool = [r for i, r in enumerate(recs) if i not in held]
    k = min(BPE_DOCS_PER_FILE, len(pool))
    sample_texts += [" ".join(r["text"].split()[:3000]) for r in random.sample(pool, k)]
    print(fname, "->", len(recs), "docs,", len(held), "held out, sampled", k)
    del recs
    gc.collect()

  t0 = time.time()
  bpe = ByteLevelBPE(vocab_size=BPE_VOCAB_SIZE).train(sample_texts)
  bpe.save(BPE_PATH)
  print(f"BPE trained in {time.time() - t0:.0f}s | vocab size: {len(bpe.itos)} | merges: {len(bpe.merges)}")

## 5. Download Training Data From Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download

train_file_paths = []
for fname in TRAIN_FILES:
  train_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=fname,
    repo_type="dataset",
    token=HF_TOKEN
  )
  train_file_paths.append(train_file_path)

print("Training files:")
for p in train_file_paths:
  print(p)

## 6. Create Train and Validation Tokens Set

Same as the starter, done file by file: every file gives `TRAIN_TOKENS / number of files` training tokens.
Each document is `<s> tokens </s>`; about 5% of the documents go to validation.
The held-out documents (512-word prompt + 128-word reference) are never used for training.
**RAM:** the files are read one document at a time (`iter_json_records`), because `json.load` of `train_high_courts.json` does not fit even in Kaggle's 30 GB. Tokens are stored in `array("H")` at 2 bytes each instead of a Python list.

In [ ]:
from array import array

def iter_json_records(path, chunk_size=16 * 1024 * 1024):
  """Reads a file that looks like [ {...}, {...}, ... ] one record at a time,
  so the whole file never has to be in memory (json.load needs all of it at once)."""
  decoder = json.JSONDecoder()
  with open(path, "r", encoding="utf-8") as f:
    buf = f.read(chunk_size)
    pos = buf.index("[") + 1
    while True:
      # skip spaces / newlines / commas between records, reading more text if needed
      while True:
        while pos < len(buf) and buf[pos] in " \r\n\t,":
          pos += 1
        if pos < len(buf):
          break
        more = f.read(chunk_size)
        if not more:
          return
        buf = more
        pos = 0
      if buf[pos] == "]":
        return
      try:
        record, end = decoder.raw_decode(buf, pos)
      except json.JSONDecodeError:
        # the record is cut off at the end of the chunk -> read more text and try again
        more = f.read(chunk_size)
        if not more:
          raise
        buf = buf[pos:] + more
        pos = 0
        continue
      yield record
      pos = end
      if pos > chunk_size:          # drop text we have already used
        buf = buf[pos:]
        pos = 0


train_tokens = array("H")   # 2 bytes per token instead of ~40 for a Python list (vocab 16000 < 65535)
val_tokens = array("H")
local_prompts, local_truth = {}, {}

train_per_file = TRAIN_TOKENS // len(TRAIN_FILES)
val_per_file = VAL_TOKENS // len(TRAIN_FILES)

random.seed(SEED)

for fname, train_file_path in zip(TRAIN_FILES, train_file_paths):
  short = fname.split("/")[-1].replace(".json", "")
  held_rng = random.Random(SEED)   # picks the held-out docs while we stream through the file
  n_held = 0
  n_docs = 0
  file_train = 0
  file_val = 0

  for i, data in enumerate(iter_json_records(train_file_path)):
    n_docs += 1
    text = data["text"]
    words = text.split()

    # held-out documents (>= 640 words) for local evaluation, never used for training
    if n_held < HELDOUT_N and len(words) >= 640 and held_rng.random() < 0.05:
      local_prompts[f"{short}_{i}"] = {"text": " ".join(words[:512])}
      local_truth[f"{short}_{i}"] = {"text": " ".join(words[512:640])}
      n_held += 1
      continue

    tokens = [bpe.bos_id] + bpe.encode_ids(text) + [bpe.eos_id]

    if random.random() < 0.05:
      if file_val < val_per_file:
        remaining = (val_per_file - file_val)
        val_tokens.extend(tokens[:remaining])
        file_val += len(tokens[:remaining])
    else:
      if file_train < train_per_file:
        remaining = (train_per_file - file_train)
        train_tokens.extend(tokens[:remaining])
        file_train += len(tokens[:remaining])

    if (file_train >= train_per_file and file_val >= val_per_file and n_held >= HELDOUT_N):
      break

  print(f"{short}: read {n_docs} docs, held out {n_held} for local evaluation")
  print(f"  train tokens from this file: {file_train:,} | val tokens: {file_val:,}")
  bpe._cache.clear()   # the BPE word cache grows with every new word; it is only a speed-up
  gc.collect()

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")

with open(LOCAL_PROMPTS_PATH, "w", encoding="utf-8") as f:
  json.dump(local_prompts, f, ensure_ascii=False)
with open(LOCAL_TRUTH_PATH, "w", encoding="utf-8") as f:
  json.dump(local_truth, f, ensure_ascii=False)
print(f"Held-out examples: {len(local_truth)}")

## 7. Create Token Blocks

In [ ]:
import numpy as np

def create_blocks(tokens, block_size):
    total_length = (len(tokens) // block_size) * block_size
    arr = np.frombuffer(tokens, dtype=np.uint16)[:total_length]
    return arr.reshape(-1, block_size).astype(np.int32)

train_blocks = create_blocks(train_tokens, TRAIN_BLOCK_SIZE)
val_blocks = create_blocks(val_tokens, TRAIN_BLOCK_SIZE)

print("Block size:", TRAIN_BLOCK_SIZE)
print("Training sequences:", len(train_blocks))
print("Validation sequences:", len(val_blocks))

## 8. Create Datasets

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"input_ids": train_blocks})
val_dataset = Dataset.from_dict({"input_ids": val_blocks})

print(train_dataset)
print(val_dataset)

del train_tokens, val_tokens, train_blocks, val_blocks   # the datasets hold their own copy
gc.collect()

## 9. Define The Model

All four models are written by hand in plain PyTorch, following Assignments 2 and 3. Every model's `forward` returns `{"loss": ..., "logits": ...}`, so the starter's `Trainer` can train it exactly like `GPT2LMHeadModel`. The loss uses the labels shifted by one position, the same as GPT-2.

**Weight tying** (Assignment 3; Module 2.4 for T5): the output layer reuses the input embedding matrix. We write `logits = x @ tok_emb.weight.T` instead of `self.head.weight = self.tok_emb.weight`. The maths is the same, but `trainer.save_model` (safetensors) cannot save two parameters that share memory.

### 9.1 Building blocks shared by `gpt` and `modern`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    """Module 2.3:  RMS(x) = sqrt( (1/E) * sum_i x_i^2 + eps ),   RMSNorm(x)_i = gamma_i * x_i / RMS(x)"""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))   # learnable, no bias (beta)

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.gamma * (x / rms)


# ---- RoPE (Module 2.3 / 3.1, Assignment 3 "rope.py") ----
def build_rope_cache(max_seq_len, d_head, base=10000.0):
    # theta_i = base^(-2(i-1)/d) for every pair of dimensions; angle = position m * theta_i
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))   # (d_head/2,)
    positions = torch.arange(max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)                                  # (T, d_head/2)
    angles = torch.cat([freqs, freqs], dim=-1)                                # (T, d_head)
    return angles.cos(), angles.sin()


def rotate_half(x):
    # split the last dim in half and return (-x2, x1)
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    # x: (B, n_heads, T, d_head); cos, sin: (B, 1, T, d_head)
    return x * cos + rotate_half(x) * sin


class CausalSelfAttention(nn.Module):
    """Masked multi-head self-attention (Module 2.3, Assignment 3).
    n_kv_heads == n_heads -> normal MHA,  n_kv_heads < n_heads -> grouped-query attention (GQA).
    past_kv = (k, v) of earlier tokens -> KV cache for generation (Module 2.3).
    attention_mask (B, S): 1 = real token, 0 = padding (Module 2.3: masks prevent attention over padding)."""
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.1, use_rope=False):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        self.attention_scale = 1.0 / math.sqrt(self.d_head)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)
        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, position_ids, attention_mask=None, past_kv=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            cos = self.rope_cos[position_ids].unsqueeze(1)   # (B, 1, T, d_head)
            sin = self.rope_sin[position_ids].unsqueeze(1)
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past_kv is not None:                  # KV cache: reuse keys/values of earlier tokens
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        new_kv = (k, v)

        if self.n_rep > 1:                       # GQA: each k/v head is shared by n_rep query heads
            k = torch.repeat_interleave(k, self.n_rep, dim=1)
            v = torch.repeat_interleave(v, self.n_rep, dim=1)

        S = k.size(2)                             # number of keys = cached + new
        scores = (q @ k.transpose(-2, -1)) * self.attention_scale        # (B, H, T, S)
        allowed = self.causal_mask[S - T:S, :S].unsqueeze(0).unsqueeze(0)  # (1, 1, T, S)
        if attention_mask is not None:
            allowed = allowed & attention_mask[:, None, None, :].bool()
            # a left-padding position has no real token before it, so let every
            # position always see itself (otherwise its whole row would be masked)
            itself = torch.eye(S, dtype=torch.bool, device=x.device)[S - T:S, :S]
            allowed = allowed | itself.unsqueeze(0).unsqueeze(0)
        scores = scores.masked_fill(~allowed, float("-inf"))
        att = F.softmax(scores, dim=-1)
        att = self.attn_dropout(att)
        out = att @ v                                                      # (B, H, T, d_head)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(out))
        return out, new_kv


class FeedForward(nn.Module):
    """Assignment 3: two linear layers with a GELU in between, plus dropout."""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class SwiGLUFeedForward(nn.Module):
    """Module 3.1 names SwiGLU as the modern feed-forward activation:
    FFN(x) = W2( SiLU(W1 x) * (W3 x) ),  SiLU(a) = a * sigmoid(a)"""
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w3 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        a = self.w1(x)
        return self.dropout(self.w2(a * torch.sigmoid(a) * self.w3(x)))

### 9.2 `gpt` and `modern`: decoder-only Transformer (Assignment 3 `DecoderLM`, pre-norm)

In [ ]:
from dataclasses import dataclass


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 640
    n_layers: int = 10
    n_heads: int = 10
    n_kv_heads: int = 10            # == n_heads for plain MHA
    d_ff: int = 2560
    max_seq_len: int = 2048
    dropout: float = 0.1
    pos_emb_type: str = "learned"   # "learned" (GPT-2 / BERT, Module 2.3) or "rope"
    norm_type: str = "layernorm"    # "layernorm" or "rmsnorm"
    ffn_type: str = "gelu"          # "gelu" or "swiglu"


class Block(nn.Module):
    """Pre-norm block (Module 2.3, used by GPT-2 and modern LLMs):  x + Attn(Norm(x)),  then  x + FFN(Norm(x))"""
    def __init__(self, cfg):
        super().__init__()
        if cfg.norm_type == "layernorm":
            self.ln1 = nn.LayerNorm(cfg.d_model)
            self.ln2 = nn.LayerNorm(cfg.d_model)
        else:
            self.ln1 = RMSNorm(cfg.d_model)
            self.ln2 = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"))
        if cfg.ffn_type == "gelu":
            self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        else:
            self.ffn = SwiGLUFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, position_ids, attention_mask=None, past_kv=None):
        a, new_kv = self.attn(self.ln1(x), position_ids, attention_mask, past_kv)
        x = x + a
        x = x + self.ffn(self.ln2(x))
        return x, new_kv


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "learned":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)    # learned absolute positions
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        if cfg.norm_type == "layernorm":
            self.ln_f = nn.LayerNorm(cfg.d_model)
        else:
            self.ln_f = RMSNorm(cfg.d_model)
        # Module 2.1 "Proper Weight Initialization": Xavier/Glorot for the embedding matrices.
        # PyTorch's default N(0, 1) embedding + weight tying gives huge first logits (start loss ~80)
        nn.init.xavier_uniform_(self.tok_emb.weight)
        if cfg.pos_emb_type == "learned":
            nn.init.xavier_uniform_(self.pos_emb.weight)

    def forward(self, input_ids, labels=None, attention_mask=None, position_ids=None,
                past_key_values=None, use_cache=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        if position_ids is None:
            position_ids = torch.arange(past_len, past_len + T, device=input_ids.device).unsqueeze(0).expand(B, T)

        x = self.tok_emb(input_ids)
        if self.cfg.pos_emb_type == "learned":
            x = x + self.pos_emb(position_ids)
        x = self.drop(x)

        new_past = []
        for i, block in enumerate(self.blocks):
            layer_past = None if past_key_values is None else past_key_values[i]
            x, kv = block(x, position_ids, attention_mask, layer_past)
            new_past.append(kv)
        x = self.ln_f(x)

        if use_cache:
            x = x[:, -1:, :]              # generation only needs the prediction for the last position
        logits = x @ self.tok_emb.weight.t()   # weight tying

        loss = None
        if labels is not None:
            # predict token t+1 from positions <= t (labels are shifted inside, like GPT-2)
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
                                   labels[:, 1:].reshape(-1))
        if use_cache:
            return {"loss": loss, "logits": logits, "past_key_values": new_past}
        return {"loss": loss, "logits": logits}

### 9.3 `rnn`: stacked LSTM / GRU language model (Module 2.1, Assignment 2 cells)

LSTM (Module 2.1, Assignment 2 packed gates):
`f, i, o = sigmoid(...)`, `g = tanh(...)`, `c_t = f*c_{t-1} + i*g`, `h_t = o*tanh(c_t)`. The forget gate bias starts at 1 (Module 2.1 tip).

GRU (Module 2.1 equations):
`z = sigmoid(W_z^h h + W_z^e e + b)`, `r = sigmoid(...)`, `h~ = tanh(W^h (r*h) + W^e e + b)`, `h_t = (1-z)*h_{t-1} + z*h~`.

Layers are stacked like Assignment 2's `RecurrentEncoder`. Dropout sits between layers (Module 2.1, "Dropout in RNNs"). Training is teacher forcing on blocks of `RNN_BLOCK_SIZE` tokens, i.e. truncated BPTT.

In [ ]:
class LSTMCell(nn.Module):
    """One time step of an LSTM (Assignment 2): packed linear layers compute f, i, g, o at once."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2g = nn.Linear(input_size, 4 * hidden_size)
        self.h2g = nn.Linear(hidden_size, 4 * hidden_size)
        with torch.no_grad():
            self.x2g.bias[0:hidden_size] = 1.0   # forget gate bias close to 1 (Module 2.1)
            self.h2g.bias[0:hidden_size] = 0.0

    def forward(self, x_t, state):
        h_prev, c_prev = state
        gates = self.x2g(x_t) + self.h2g(h_prev)
        f_pre, i_pre, g_pre, o_pre = gates.chunk(4, dim=-1)
        f_t = torch.sigmoid(f_pre)
        i_t = torch.sigmoid(i_pre)
        g_t = torch.tanh(g_pre)
        o_t = torch.sigmoid(o_pre)
        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * torch.tanh(c_t)
        return (h_t, c_t)


class GRUCell(nn.Module):
    """One time step of a GRU with the Module 2.1 equations (layer names as in Assignment 2)."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x2z = nn.Linear(input_size, hidden_size)
        self.h2z = nn.Linear(hidden_size, hidden_size, bias=False)
        self.x2r = nn.Linear(input_size, hidden_size)
        self.h2r = nn.Linear(hidden_size, hidden_size, bias=False)
        self.x2n = nn.Linear(input_size, hidden_size)
        self.h2n = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x_t, h_prev):
        z_t = torch.sigmoid(self.x2z(x_t) + self.h2z(h_prev))          # update gate
        r_t = torch.sigmoid(self.x2r(x_t) + self.h2r(h_prev))          # reset gate
        n_t = torch.tanh(self.x2n(x_t) + self.h2n(r_t * h_prev))       # candidate h~
        h_t = (1 - z_t) * h_prev + z_t * n_t
        return h_t


class RNNLM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers, cell_type="lstm", dropout=0.1):
        super().__init__()
        self.is_lstm = (cell_type == "lstm")
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.tok_emb = nn.Embedding(vocab_size, hidden_size)
        cell_cls = LSTMCell if self.is_lstm else GRUCell
        self.cells = nn.ModuleList([cell_cls(hidden_size, hidden_size) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.tok_emb.weight)     # Module 2.1: Xavier/Glorot initialisation

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None, use_cache=False):
        """past_key_values here = list of per-layer states (h, c) / h, carried during generation.
        attention_mask (B, T): at padding positions the state is NOT updated."""
        B, T = input_ids.shape
        x = self.dropout(self.tok_emb(input_ids))
        new_states = []
        for l in range(self.num_layers):
            cell = self.cells[l]
            if past_key_values is not None:
                state = past_key_values[l]
            else:
                h = torch.zeros(B, self.hidden_size, device=x.device, dtype=x.dtype)
                state = (h, torch.zeros_like(h)) if self.is_lstm else h
            outputs = []
            for t in range(T):           # explicit unroll over time (BPTT through these steps)
                new_state = cell(x[:, t, :], state)
                if attention_mask is not None:
                    m = attention_mask[:, t].unsqueeze(1).to(x.dtype)     # 1 = real token, 0 = pad
                    if self.is_lstm:
                        new_state = (m * new_state[0] + (1 - m) * state[0], m * new_state[1] + (1 - m) * state[1])
                    else:
                        new_state = m * new_state + (1 - m) * state
                state = new_state
                outputs.append(state[0] if self.is_lstm else state)
            new_states.append(state)
            x = self.dropout(torch.stack(outputs, dim=1))    # input of the next layer

        if use_cache:
            x = x[:, -1:, :]
        logits = x @ self.tok_emb.weight.t()       # weight tying (Assignment 3)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1, :].reshape(-1, logits.size(-1)).float(),
                                   labels[:, 1:].reshape(-1))
        if use_cache:
            return {"loss": loss, "logits": logits, "past_key_values": new_states}
        return {"loss": loss, "logits": logits}

### 9.4 `t5`: encoder-decoder Transformer (Module 2.3 and Module 2.4)

From the T5 slides:
* Normalisation sits outside the residual path, with no bias (RMSNorm).
* Relative position bias: 32 learned scalars per head, shared by all layers of a stack. Buckets grow logarithmically up to offset 128.
* Dropout in the FFN, on residuals, on attention weights, and at the input/output of each stack.
* The decoder uses causal self-attention plus cross-attention, where queries come from the decoder and keys/values from the encoder.
* The input embedding and output projection share one matrix.

Pre-training uses span corruption with sentinel tokens. Fine-tuning is teacher forcing on prompt → continuation.

In [ ]:
class T5RelativePositionBias(nn.Module):
    def __init__(self, n_heads, bidirectional, num_buckets=32, max_distance=128):
        super().__init__()
        self.bidirectional = bidirectional
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        self.rel_emb = nn.Embedding(num_buckets, n_heads)     # one scalar per (bucket, head)

    def bucket(self, relative_position):
        # relative_position = key_pos - query_pos
        n = -relative_position
        num_buckets = self.num_buckets
        ret = torch.zeros_like(n)
        if self.bidirectional:                 # encoder: half the buckets for keys on the right
            num_buckets = num_buckets // 2
            ret = ret + (n < 0).long() * num_buckets
            n = torch.abs(n)
        else:                                  # decoder: only keys on the left exist
            n = torch.clamp(n, min=0)
        max_exact = num_buckets // 2           # small offsets get their own bucket
        is_small = n < max_exact
        # bigger offsets: logarithmically wider buckets, everything beyond max_distance shares the last one
        large = max_exact + (torch.log(torch.clamp(n, min=1).float() / max_exact)
                             / math.log(self.max_distance / max_exact)
                             * (num_buckets - max_exact)).long()
        large = torch.clamp(large, max=num_buckets - 1)
        return ret + torch.where(is_small, n, large)

    def forward(self, q_len, k_len, device):
        q_pos = torch.arange(q_len, device=device)[:, None]
        k_pos = torch.arange(k_len, device=device)[None, :]
        buckets = self.bucket(k_pos - q_pos)               # (q, k)
        bias = self.rel_emb(buckets)                       # (q, k, H)
        return bias.permute(2, 0, 1).unsqueeze(0)          # (1, H, q, k)


class T5Attention(nn.Module):
    """Self-attention (x_kv = x_q) or cross-attention (x_kv = encoder output)."""
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x_q, x_kv, position_bias=None, allowed=None):
        B, Tq, C = x_q.shape
        Tk = x_kv.size(1)
        q = self.q_proj(x_q).view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) * self.attention_scale
        if position_bias is not None:
            scores = scores + position_bias
        if allowed is not None:
            scores = scores.masked_fill(~allowed, float("-inf"))
        att = self.attn_dropout(F.softmax(scores, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, Tq, C)
        return self.out_proj(out)


class T5FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))   # dropout inside the FFN


class T5EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = T5Attention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = T5FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, position_bias, src_allowed):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, position_bias, src_allowed))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class T5DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.self_attn = T5Attention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.cross_attn = T5Attention(d_model, n_heads, dropout)
        self.norm3 = RMSNorm(d_model)
        self.ffn = T5FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, position_bias, causal_allowed, src_allowed):
        h = self.norm1(x)
        x = x + self.dropout(self.self_attn(h, h, position_bias, causal_allowed))   # masked self-attention
        h = self.norm2(x)
        x = x + self.dropout(self.cross_attn(h, enc_out, None, src_allowed))        # cross-attention
        x = x + self.dropout(self.ffn(self.norm3(x)))
        return x


class T5Model(nn.Module):
    def __init__(self, vocab_size, pad_id, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.shared_emb = nn.Embedding(vocab_size, d_model)    # encoder input, decoder input and output
        self.dropout = nn.Dropout(dropout)
        self.enc_layers = nn.ModuleList([T5EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)])
        self.dec_layers = nn.ModuleList([T5DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_dec_layers)])
        self.enc_norm = RMSNorm(d_model)
        self.dec_norm = RMSNorm(d_model)
        self.enc_pos_bias = T5RelativePositionBias(n_heads, bidirectional=True)
        self.dec_pos_bias = T5RelativePositionBias(n_heads, bidirectional=False)
        nn.init.xavier_uniform_(self.shared_emb.weight)  # Module 2.1: Xavier/Glorot initialisation

    def encode(self, input_ids):
        S = input_ids.size(1)
        src_allowed = (input_ids != self.pad_id)[:, None, None, :]          # no attention to padding
        position_bias = self.enc_pos_bias(S, S, input_ids.device)
        x = self.dropout(self.shared_emb(input_ids))
        for layer in self.enc_layers:
            x = layer(x, position_bias, src_allowed)
        return self.dropout(self.enc_norm(x))

    def decode(self, decoder_input_ids, enc_out, input_ids, last_only=False):
        T = decoder_input_ids.size(1)
        src_allowed = (input_ids != self.pad_id)[:, None, None, :]
        causal_allowed = torch.tril(torch.ones(T, T, dtype=torch.bool, device=enc_out.device))[None, None]
        position_bias = self.dec_pos_bias(T, T, enc_out.device)
        x = self.dropout(self.shared_emb(decoder_input_ids))
        for layer in self.dec_layers:
            x = layer(x, enc_out, position_bias, causal_allowed, src_allowed)
        x = self.dropout(self.dec_norm(x))
        if last_only:
            x = x[:, -1:, :]
        return x @ self.shared_emb.weight.t()                                # weight sharing

    def forward(self, input_ids, decoder_input_ids, labels=None):
        enc_out = self.encode(input_ids)
        logits = self.decode(decoder_input_ids, enc_out, input_ids)
        loss = None
        if labels is not None:
            # decoder_input_ids are already shifted (<s> + target[:-1]); -100 = padding, ignored
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), labels.reshape(-1), ignore_index=-100)
        return {"loss": loss, "logits": logits}

### 9.5 Build the model chosen by `MODEL_TYPE`

In [ ]:
def build_model():
  V = len(bpe.itos)
  if MODEL_TYPE == "gpt":
    cfg = LMConfig(vocab_size=V, d_model=N_EMBD, n_layers=N_LAYER, n_heads=N_HEAD, n_kv_heads=N_HEAD,
                   d_ff=D_FF, max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT,
                   pos_emb_type="learned", norm_type="layernorm", ffn_type="gelu")
    return DecoderLM(cfg)
  if MODEL_TYPE == "modern":
    cfg = LMConfig(vocab_size=V, d_model=N_EMBD, n_layers=N_LAYER, n_heads=N_HEAD, n_kv_heads=N_KV_HEAD,
                   d_ff=D_FF, max_seq_len=MODEL_CONTEXT_LENGTH, dropout=DROPOUT,
                   pos_emb_type="rope", norm_type="rmsnorm", ffn_type="swiglu")
    return DecoderLM(cfg)
  if MODEL_TYPE == "rnn":
    return RNNLM(V, RNN_HIDDEN, RNN_LAYERS, cell_type=RNN_CELL, dropout=DROPOUT)
  if MODEL_TYPE == "t5":
    return T5Model(V + T5_N_SENTINELS, bpe.pad_id, T5_D_MODEL, T5_ENC_LAYERS, T5_DEC_LAYERS,
                   T5_HEADS, T5_D_FF, dropout=DROPOUT)

model = build_model()

model_size = sum(p.numel() for p in model.parameters())
print(f"{MODEL_TYPE}: Model size: {model_size / 1e6:.2f}M parameters")
if not SMOKE_TEST and not (50e6 <= model_size <= 300e6):
  raise ValueError(
    f"{model_size/1e6:.1f}M parameters is outside the allowed 50M-300M range. "
    "Change the sizes in the configuration cell."
  )

### 9.6 Data collators

`gpt`, `modern` and `rnn` use the starter's collator: `labels = input_ids`, and the model shifts them.

`t5` turns each block into an encoder input and a decoder target:
* **Continuation** (fine-tuning): the encoder gets the first `T5_ENC_LEN` tokens, and the decoder must write the next `T5_DEC_LEN` tokens.
* **Span corruption** (pre-training, Module 2.4): 15% of the tokens are corrupted in spans with mean length 3. Each span becomes one sentinel in the input, and the target is `sentinel span sentinel span ... final sentinel`.

In [ ]:
def data_collator(features):
  # blocks are all exactly TRAIN_BLOCK_SIZE long, so no padding is needed;
  # the model shifts the labels internally (like GPT2LMHeadModel)
  ids = torch.tensor([f["input_ids"] for f in features], dtype=torch.long)
  return {"input_ids": ids, "labels": ids.clone()}


def t5_continuation_collator(features):
  src, dec_in, labels = [], [], []
  for f in features:
    block = f["input_ids"]
    target = block[T5_ENC_LEN:T5_ENC_LEN + T5_DEC_LEN]
    src.append(block[:T5_ENC_LEN])
    dec_in.append([bpe.bos_id] + target[:-1])      # teacher forcing: decoder input = target shifted right
    labels.append(target)
  return {"input_ids": torch.tensor(src, dtype=torch.long),
          "decoder_input_ids": torch.tensor(dec_in, dtype=torch.long),
          "labels": torch.tensor(labels, dtype=torch.long)}


def make_span_corruption(tokens, noise_density=0.15, mean_span_length=3):
  sentinel_start = len(bpe.itos)                      # sentinel k has id len(vocab) + k
  start_prob = noise_density / mean_span_length
  src, tgt = [], []
  i = 0
  k = 0
  while i < len(tokens):
    if k < T5_N_SENTINELS - 1 and random.random() < start_prob:
      span = random.randint(1, 2 * mean_span_length - 1)   # 1..5, mean 3
      span = min(span, len(tokens) - i)
      src.append(sentinel_start + k)
      tgt.append(sentinel_start + k)
      tgt.extend(tokens[i:i + span])
      i += span
      k += 1
    else:
      src.append(tokens[i])
      i += 1
  tgt.append(sentinel_start + k)                       # final sentinel
  return src, tgt


def t5_span_collator(features):
  src, dec_in, labels = [], [], []
  for f in features:
    s, t = make_span_corruption(f["input_ids"][:T5_ENC_LEN])
    t = t[:T5_DEC_LEN]
    d = [bpe.bos_id] + t[:-1]
    src.append(s + [bpe.pad_id] * (T5_ENC_LEN - len(s)))
    dec_in.append(d + [bpe.pad_id] * (T5_DEC_LEN - len(d)))
    labels.append(t + [-100] * (T5_DEC_LEN - len(t)))     # -100 = ignored by the loss
  return {"input_ids": torch.tensor(src, dtype=torch.long),
          "decoder_input_ids": torch.tensor(dec_in, dtype=torch.long),
          "labels": torch.tensor(labels, dtype=torch.long)}

## 10. Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
  output_dir=MODEL_OUTPUT_DIR,
  num_train_epochs=EPOCHS,
  per_device_train_batch_size=BATCH_SIZE,
  per_device_eval_batch_size=BATCH_SIZE,
  gradient_accumulation_steps=GRAD_ACCUM,
  learning_rate=LEARNING_RATE,
  weight_decay=0.1,
  warmup_steps=10 if SMOKE_TEST else 500,
  max_steps=100 if SMOKE_TEST else -1,
  lr_scheduler_type="cosine",
  max_grad_norm=MAX_GRAD_NORM,
  fp16=torch.cuda.is_available(),
  eval_strategy="steps",
  eval_steps=50 if SMOKE_TEST else 1000,
  logging_steps=10 if SMOKE_TEST else 100,
  save_steps=50 if SMOKE_TEST else 1000,
  save_total_limit=2,
  seed=SEED,
  report_to="none",
)

if MODEL_TYPE == "t5":
  # stage 2 of the T5 pipeline (Module 2.4): fine-tuning with a constant learning rate
  finetune_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR + "-finetune",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=T5_FINETUNE_LR,
    weight_decay=0.1,
    max_steps=100 if SMOKE_TEST else -1,
    lr_scheduler_type="constant",
    max_grad_norm=MAX_GRAD_NORM,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=50 if SMOKE_TEST else 1000,
    logging_steps=10 if SMOKE_TEST else 100,
    save_steps=50 if SMOKE_TEST else 1000,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
  )

## 11. Training

For `t5` there are two stages (Module 2.4 T5 pipeline):
1. **Pre-training:** span corruption on the first `T5_SPAN_FRACTION` of the training blocks.
2. **Fine-tuning:** teacher forcing on prompt → continuation, using the rest of the blocks.

The validation loss reported afterwards is the continuation loss.

In [ ]:
from transformers import Trainer

start_time = time.time()

if MODEL_TYPE == "t5":
  n_span = int(len(train_dataset) * T5_SPAN_FRACTION)
  span_dataset = train_dataset.select(range(n_span))
  continuation_dataset = train_dataset.select(range(n_span, len(train_dataset)))

  print("T5 stage 1: span-corruption pre-training on", len(span_dataset), "blocks")
  pretrain_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=span_dataset,
    eval_dataset=val_dataset,
    data_collator=t5_span_collator
  )
  pretrain_trainer.train()

  print("T5 stage 2: prompt -> continuation fine-tuning on", len(continuation_dataset), "blocks")
  trainer = Trainer(
    model=model,
    args=finetune_args,
    train_dataset=continuation_dataset,
    eval_dataset=val_dataset,
    data_collator=t5_continuation_collator
  )
  trainer.train()
else:
  trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
  )
  trainer.train()

training_time = (time.time() - start_time)

tokens_per_epoch = (len(train_dataset) * TRAIN_BLOCK_SIZE)

total_tokens = (tokens_per_epoch * EPOCHS)

tokens_per_second = (total_tokens / training_time)

print(f"Training time: {training_time / 3600:.2f} hours")
print(f"Training speed: {tokens_per_second:,.0f} tokens/sec")

## 12. Perplexity

In [ ]:
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)
print(f"Validation loss: {eval_loss:.6f}")
print(f"Validation perplexity: {perplexity:.4f}")

## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
if MODEL_TYPE == "t5":
  # stage 1 + stage 2 on one time axis
  offset = pretrain_trainer.state.global_step
  logs = pretrain_trainer.state.log_history + [dict(l, step=l["step"] + offset) for l in trainer.state.log_history if "step" in l]

train_steps = []
train_loss = []
eval_steps = []
eval_loss = []

for log in logs:
  if ("loss" in log and "step" in log):
    train_steps.append(log["step"])
    train_loss.append(log["loss"])
  if ("eval_loss" in log and "step" in log):
    eval_steps.append(log["step"])
    eval_loss.append(log["eval_loss"])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss, label="Training Loss")
plt.plot(eval_steps, eval_loss, label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title(f"Phase 1 Training and Validation Loss ({MODEL_TYPE})")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

print("Loss plot saved to:")
print(LOSS_PLOT_PATH)

## 14. Save Final Model

In [ ]:
trainer.save_model(MODEL_OUTPUT_DIR)
bpe.save(os.path.join(MODEL_OUTPUT_DIR, "bpe_legal.json"))

print("Final model saved to:")
print(MODEL_OUTPUT_DIR)

## 15. Push Model to Hugging Face Hub

In [ ]:
# model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)

# print("Model and tokenizer pushed successfully.")

## 16. Load Test Data from HuggingFace

In [ ]:
test_file_path = hf_hub_download(
  repo_id=HF_DATASET_REPO,
  filename=TEST_FILE,
  repo_type="dataset",
  token=HF_TOKEN
)

print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
  test_data = json.load(f)

print("Number of test examples:", len(test_data))

## 18. Load Final Model for Generation
`trainer.save_model` only saves the weights of a model that is not a Hugging Face `PreTrainedModel`. So we rebuild the same architecture and load them.

In [ ]:
from safetensors.torch import load_file

generation_bpe = ByteLevelBPE.load_bpe(os.path.join(MODEL_OUTPUT_DIR, "bpe_legal.json"))
generation_model = build_model()
weights_path = os.path.join(MODEL_OUTPUT_DIR, "model.safetensors")
if os.path.exists(weights_path):
  generation_model.load_state_dict(load_file(weights_path))
else:
  generation_model.load_state_dict(torch.load(os.path.join(MODEL_OUTPUT_DIR, "pytorch_model.bin"), map_location="cpu"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generation_model = generation_model.to(device)
generation_model.eval()

print("Device:", device)
print(f"Parameters: {sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M")

## 19. Generate 128 Words
#### (words, not tokens)
This replaces `model.generate`. It does the same job:
* left padding plus an attention mask, as in the starter,
* reading the prompt once, then adding one token at a time with a KV cache (Module 2.3), or for the RNN by carrying the hidden state,
* stopping at `</s>`, then cutting the text to 128 words.

The next token is picked with the decoding strategies from Module 2.1 and Assignment 2: greedy, sampling with temperature, top-k, or top-p.

In [ ]:
def sample_next_token(logits, strategy="top_p", temperature=1.0, top_k=10, top_p=0.9):
  """Assignment 2 _sample_attn_next_token. logits: (B, V) -> (B,) token ids"""
  if temperature <= 0:
    raise ValueError("temperature must be > 0")
  if strategy == "greedy":
    return torch.argmax(logits, dim=-1)
  probs = F.softmax(logits / temperature, dim=-1)
  if strategy == "sample":
    return torch.multinomial(probs, num_samples=1).squeeze(1)
  if strategy == "top_k":
    if top_k < 1:
      raise ValueError("top_k must be >= 1")
    k = min(top_k, probs.size(-1))
    values, indices = torch.topk(probs, k, dim=-1)
    sampled_local = torch.multinomial(values, num_samples=1)        # multinomial re-normalises
    return torch.gather(indices, -1, sampled_local).squeeze(1)
  if strategy == "top_p":
    if not (0 < top_p <= 1):
      raise ValueError("top_p must be in (0, 1]")
    sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
    keep = (torch.cumsum(sorted_probs, dim=-1) - sorted_probs) < top_p   # smallest set with mass >= top_p
    keep[:, 0] = True                                                    # always keep the most likely token
    filtered = sorted_probs * keep
    sampled_local = torch.multinomial(filtered, num_samples=1)
    return torch.gather(sorted_indices, -1, sampled_local).squeeze(1)
  raise ValueError("unknown decoding strategy " + strategy)


def left_pad(seqs, pad_id):
  L = max(len(s) for s in seqs)
  input_ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
  attention_mask = torch.zeros((len(seqs), L), dtype=torch.long)
  for i, s in enumerate(seqs):                 # LEFT padding (like the starter)
    input_ids[i, L - len(s):] = torch.tensor(s, dtype=torch.long)
    attention_mask[i, L - len(s):] = 1
  return input_ids, attention_mask


def generate_batch(prompts, model, bpe, device, target_words=128):
  V = len(bpe.itos)
  if MODEL_TYPE == "t5":
    max_prompt = T5_ENC_LEN
  else:
    max_prompt = MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS

  seqs = []
  for p in prompts:
    ids = bpe.encode_ids(p)
    seqs.append([bpe.bos_id] + ids[-(max_prompt - 1):])   # keep the end of the prompt
  input_ids, attention_mask = left_pad(seqs, bpe.pad_id)
  input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
  B = input_ids.size(0)

  generated = []
  finished = torch.zeros(B, dtype=torch.bool, device=device)

  with torch.inference_mode():
    if MODEL_TYPE == "t5":
      enc_out = model.encode(input_ids)
      dec_in = torch.full((B, 1), bpe.bos_id, dtype=torch.long, device=device)
      for step in range(MAX_GENERATION_TOKENS):
        logits = model.decode(dec_in, enc_out, input_ids, last_only=True)[:, -1, :].float()
        next_token = sample_next_token(logits, DECODING_STRATEGY, TEMPERATURE, TOP_K, TOP_P)
        next_token = torch.where(finished, torch.full_like(next_token, bpe.pad_id), next_token)
        generated.append(next_token)
        finished = finished | (next_token == bpe.eos_id)
        if finished.all():
          break
        dec_in = torch.cat([dec_in, next_token.unsqueeze(1)], dim=1)
    else:
      if MODEL_TYPE == "rnn":
        out = model(input_ids, attention_mask=attention_mask, use_cache=True)
      else:
        position_ids = (attention_mask.cumsum(dim=-1) - 1).clamp(min=0)   # positions start at the first real token
        out = model(input_ids, attention_mask=attention_mask, position_ids=position_ids, use_cache=True)
        next_pos = position_ids[:, -1:] + 1
      past = out["past_key_values"]
      logits = out["logits"][:, -1, :].float()
      for step in range(MAX_GENERATION_TOKENS):
        next_token = sample_next_token(logits, DECODING_STRATEGY, TEMPERATURE, TOP_K, TOP_P)
        next_token = torch.where(finished, torch.full_like(next_token, bpe.pad_id), next_token)
        generated.append(next_token)
        finished = finished | (next_token == bpe.eos_id)
        if finished.all() or step == MAX_GENERATION_TOKENS - 1:
          break
        attention_mask = torch.cat([attention_mask, torch.ones((B, 1), dtype=attention_mask.dtype, device=device)], dim=1)
        if MODEL_TYPE == "rnn":
          out = model(next_token.unsqueeze(1), past_key_values=past, use_cache=True)
        else:
          out = model(next_token.unsqueeze(1), attention_mask=attention_mask, position_ids=next_pos,
                      past_key_values=past, use_cache=True)
          next_pos = next_pos + 1
        past = out["past_key_values"]
        logits = out["logits"][:, -1, :].float()

  generated = torch.stack(generated, dim=1).tolist()      # new tokens only
  predictions = []
  for row in generated:
    if bpe.eos_id in row:
      row = row[:row.index(bpe.eos_id)]
    row = [t for t in row if t < V]                       # drop T5 sentinel ids (not real text)
    words = bpe.decode(row).split()[:target_words]
    predictions.append(" ".join(words))
  return predictions

## 20. Generate Predictions for the Test Set

In [ ]:
predictions = {}

items = list(test_data.items())
if SMOKE_TEST:
  items = items[:32]
total_examples = len(items)
print(f"Total test examples: {total_examples}")

for start in range(0, total_examples, GENERATION_BATCH_SIZE):
  batch = items[start:start + GENERATION_BATCH_SIZE]
  batch_ids = [str(example_id) for example_id, _ in batch]
  batch_prompts = [record["text"] for _, record in batch]

  batch_predictions = generate_batch(
  prompts=batch_prompts,
  model=generation_model,
  bpe=generation_bpe,
  device=device,
  target_words=128
  )

  for example_id, prediction in zip(batch_ids, batch_predictions):
    predictions[example_id] = {"text": prediction}

  completed = start + len(batch)
  print(f"Completed {completed}/{total_examples} ({100 * completed / total_examples:.1f}%)")

## 21. Save prediction.json

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
  json.dump(
  predictions,
  f,
  ensure_ascii=False,
  indent=2
  )

print("prediction json saved:")
print(PREDICTION_PATH)

## 22. Create CodaBench Submission ZIP

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
  zipf.write(PREDICTION_PATH, arcname="prediction.json")   # the file inside the zip must be prediction.json

print("Codabench submission created:")
print(SUBMISSION_ZIP)

## 23. Local evaluation on the held-out documents

In [ ]:
!pip install bert-score rouge-score scikit-learn

In [ ]:
local_preds = {}
local_ids = list(local_prompts.keys())
for start in range(0, len(local_ids), GENERATION_BATCH_SIZE):
    chunk = local_ids[start:start + GENERATION_BATCH_SIZE]
    outs = generate_batch(
        [local_prompts[i]["text"] for i in chunk],
        generation_model, generation_bpe, device, target_words=128
    )
    for i, o in zip(chunk, outs):
        local_preds[i] = {"text": o if o.strip() else "the"}   # the evaluator needs non-empty text

with open(LOCAL_PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(local_preds, f, ensure_ascii=False)

# trivial baseline: repeat the last 128 words of each prompt
baseline = {i: {"text": " ".join(local_prompts[i]["text"].split()[-128:])} for i in local_prompts}
with open(LOCAL_BASELINE_PATH, "w", encoding="utf-8") as f:
    json.dump(baseline, f, ensure_ascii=False)

print("Saved:", LOCAL_PREDICTION_PATH, "and", LOCAL_BASELINE_PATH)
for i in local_ids[:2]:
    print("\n--- held-out", i, "---")
    print("PROMPT END :", " ".join(local_prompts[i]["text"].split()[-30:]))
    print("MODEL      :", local_preds[i]["text"][:300])
    print("REFERENCE  :", local_truth[i]["text"][:300])

In [ ]:
import glob, subprocess, sys

# Put evaluate_phase1.py (from the organisers' evaluation-scripts folder) next to the notebook
# or in OUTPUT_DIR, and `pip install` what its requirements.txt lists first.
found = glob.glob("/content/evaluate_phase1.py") + glob.glob(os.path.join(OUTPUT_DIR, "evaluate_phase1.py"))
if not found:
    print("evaluate_phase1.py not found - add it, then re-run this cell.")
else:
    for name, path in [("MODEL " + MODEL_TYPE, LOCAL_PREDICTION_PATH), ("BASELINE (last 128 prompt words)", LOCAL_BASELINE_PATH)]:
        print("=====", name, "=====")
        r = subprocess.run([sys.executable, found[0], "--truth", LOCAL_TRUTH_PATH, "--prediction", path],
                           capture_output=True, text=True)
        print(r.stdout or r.stderr)